In [1]:
%load_ext autoreload
%autoreload 2

import os 
import sys

import pandas as pd
import numpy as np

from preproces_prod3 import *
    
local_path = os.getcwd()
code_root = os.path.abspath(os.path.join(local_path, '..', 'Code'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)
code_root = os.path.abspath(os.path.join(local_path, '../..', 'inv'))

if code_root not in sys.path:
    sys.path.insert(0, code_root)

from matching_case_control import *
#all_data,analyze_vrs_data, integer_programming_matching_gurobi,match_nn_max_dist_weigths,comparar_medias_test, charly_mip, charly_double_mip, mylogit, results_match, tabla_marcel, tabla_final,summary_eff, summary_eff_aux

import inv

from IPython.core.display import display
display.max_output_lines = 500  # Adjust the number as needed
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

np.random.seed(42)

path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

df_vrs_match_case = pd.read_csv(path_data/'df_vrs_match_case_11_04_25.csv', index_col=0)  #df_vrs_match_case_s39_2012
df_upc_match_case = pd.read_csv(path_data/'df_upc_match_case_11_04_25.csv', index_col=0) #df_upc_match_case_s39_2012

icd10_codes_fr = [
    "N390",   # Infección del tracto urinario (sin síntomas respiratorios) INFECCION DE VÍAS URINARIAS, SITIO NO ESPECIFICADO
    "A090",     # Gastroenteritis aguda (sin síntomas respiratorios) OTRAS GASTROENTERITIS Y COLITIS NO ESPECIFICADAS DE ORIGEN INFECCIOSO
    "A099",     # Gastroenteritis aguda (sin síntomas respiratorios) GASTROENTERITIS Y COLITIS DE ORIGEN NO ESPECIFICADO
    "A09X",     # Gastroenteritis aguda (sin síntomas respiratorios) DIARREA Y GASTROENTERITIS DE PRESUNO ORIGEN INFECCIOSO
    "R100",  # Cólico infantil (sin fiebre ni síntomas respiratorios) ABDOMEN AGUDO
    "R101",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR ABDOMINAL LOCALIZADO EN PARTE SUPERIOR
    "R102",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR PELVICO Y PERINEAL
    "R103",  # Cólico infantil (sin fiebre ni síntomas respiratorios) DOLOR LOCALIZADO EN OTRAS PARTES INFERIORES DEL ABDOMEN
    "R104",  # Cólico infantil (sin fiebre ni síntomas respiratorios) OTROS DOLORES ABDOMINALES Y LOS NO ESPECIFICADOS
    "R11X",  # Cólico infantil (sin fiebre ni síntomas respiratorios) NAUSEA Y VOMITO
    "R634",   # Pérdida de peso (sin fiebre ni síntomas respiratorios) PERDIDA ANORMAL DE PESO
    "R633",   # Dificultades de alimentación (sin fiebre ni síntomas respiratorios) DIFICULTADES Y MALA ADMINISTRACION DE LA ALIMENTACION
    "P599",   # Ictericia neonatal (sin fiebre ni síntomas respiratorios) ICTERICIA NEONATAL, NO ESPECIFICADA
    "R681",  # Llanto anormal (sin fiebre ni síntomas respiratorios) SINTOMAS NO ESPECIFICOS PROPIOS DE LA INFANCIA
    "S099",  # Traumatismo craneal (sin fiebre ni síntomas respiratorios)
    "Z539"    # Cirugía de emergencia (sin fiebre ni síntomas respiratorios)
    ################### AÑADIDOS POR MI #######################
    'A099', 
    'A080', 
    'A083', 
    'A082', 
    'A084',
    'A090', 
    'A081', 
    'A085'
]

ruts_cariopaticos_1 = pd.read_csv(path_data/'lista_pacientes_riesgo.csv').query('card1')
ruts_cariopaticos_2 = pd.read_csv(path_data/'lista_pacientes_riesgo.csv').query('card2')
ruts_displasia = pd.read_csv(path_data/'lista_pacientes_riesgo.csv').query('displ')

list_experiments=[]
df = (df_vrs_match_case
      .drop(columns=[col for col in df_vrs_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
      .assign(cardio1 = lambda x: x.RUN.isin(ruts_cariopaticos_1.RUN).astype(int),
              displa = lambda x: x.RUN.isin(ruts_displasia.RUN).astype(int),)
      .copy()
      )

df_upc = (
    df_upc_match_case
    .drop(columns=[col for col in df_upc_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
    .assign(cardio1 = lambda x: x.RUN.isin(ruts_cariopaticos_1.RUN).astype(int),
              displa = lambda x: x.RUN.isin(ruts_displasia.RUN).astype(int),)
    .copy()
)

df.loc[df.RUN == 'ac483764636448d753930868dd3192a785e3728a2d81b3fc44dba45c3506255a', 'region'] = 'METROPOLITANA'
df.loc[df.COMUNA == 12202, 'region'] = 'MAGALLANES Y ANTARTICA'


In [8]:
df_all_vrs, _ = call_data('NAC_RNI_EGRESOS_ENTREGA_ISCI_11_04_2025_encr.csv')


n_rows_inicial= 157557
['DESCONOCIDO---->None' 'Región De Antofagasta---->ANTOFAGASTA'
 'Región De Arica Parinacota---->ARICA Y PARINACOTA'
 'Región De Atacama---->ATACAMA'
 'Región De Aysén del General Carlos Ibañez del Campo---->AISEN'
 'Región De Coquimbo---->COQUIMBO' 'Región De La Araucanía---->ARAUCANIA'
 'Región De Los Lagos---->LOS LAGOS' 'Región De Los Ríos---->LOS RIOS'
 'Región De Magallanes y de la Antártica Chilena---->MAGALLANES Y ANTARTICA'
 'Región De Tarapacá---->TARAPACA' 'Región De Valparaíso---->VALPARAISO'
 'Región De Ñuble---->NUBLE' 'Región Del Bíobío---->BIOBIO'
 "Región Del Libertador Gral. B. O'Higgins---->O'HIGGINS"
 'Región Del Maule---->MAULE'
 'Región Metropolitana de Santiago---->METROPOLITANA']
n_rows_post_prefiltred= 157557
Datos perdidos por muertes:  1227
ruts perdidos por filtro semanas y peso:  491
Droped intersex: 14
Datos perdidos por edad madre atípica: 2
Datos perdidos por fecha ingreso menor a fecha nacimiento: 19
vrs en los primeros 7 dias de 

In [9]:
with open(path_data/'lista_ruts_cardio.pkl', 'rb') as f:
    lista_ruts_cardio = pickle.load(f)

df = (df_all_vrs
      .drop(columns=[col for col in df_vrs_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
      .assign(cardio1 = lambda x: x.RUN.isin(lista_ruts_cardio).astype(int),
              displa = lambda x: x.RUN.isin(ruts_displasia.RUN).astype(int),)
      .copy()
      )
df.loc[df.RUN == 'ac483764636448d753930868dd3192a785e3728a2d81b3fc44dba45c3506255a', 'region'] = 'METROPOLITANA'
df.loc[df.COMUNA == 12202, 'region'] = 'MAGALLANES Y ANTARTICA'

In [63]:
df_match_problem = pd.read_csv(path_data/'matched_data_cardiopats_1_seasonal_francia.csv',index_col=0)[['RUN','SEMANAS','PESO','event','estado_inmunizacion']]
varianzas = df_match_problem.var(numeric_only=True)
print(varianzas)

SEMANAS                     0.266667
PESO                   240376.666667
event                       0.300000
estado_inmunizacion         0.166667
dtype: float64


In [58]:
df_match_problem

,RUN,SEMANAS,PESO,event,inmunizado
0,28b5a2ba1aa014a570be38c4cfa8448b8f50717de5161538d7b00c9d67cdc54e,38.0,3140.0,1,1
1,823ab416a40c436d7a1fa4489430f62873edd04fe11334ae4a5131848b08d75f,38.0,2605.0,1,1
2,ec3dd7c528f9103694e305aa7a1881e232237ef903838918610b9939630fe05e,39.0,3325.0,1,0
3,c85379ecfc5d576486dc39abadf46246f54c1b2971b224bd5762ff3c475285b3,38.0,3900.0,0,1
4,bf16d7063d946dba531b63ecacc9b811df1fe1b07879abca009b1373789e7325,38.0,3900.0,0,1
5,48db718f95939cd0b7ccde47d088a9c3a54c1a98ce375339762ba3353eb8cb2a,39.0,3370.0,0,1


In [59]:
df_match_problem.corr(numeric_only=True)

,SEMANAS,PESO,event,inmunizado
SEMANAS,1.000000e+00,-0.040814,2.276634e-15,-0.632456
PESO,-4.081407e-02,1.000000,-7.820104e-01,0.048295
event,2.276634e-15,-0.782010,1.000000e+00,-0.447214
inmunizado,-6.324555e-01,0.048295,-4.472136e-01,1.000000


In [60]:
print(df_match_problem.shape)

(6, 5)


In [64]:
pd.crosstab(df_match_problem['event'], df_match_problem['estado_inmunizacion'])

estado_inmunizacion,0,1
event,,
0,0,3
1,1,2


In [67]:
contingency_table = pd.crosstab(
    df_match_problem['event'], #'diag_vrs' cama_bin
    df_match_problem['estado_inmunizacion'], 
    rownames=['Diagnóstico VRS'], 
    colnames=['Estado de Inmunización'], 
    margins=True
)

table_clean = contingency_table.drop(index='All', errors='ignore').drop(columns='All', errors='ignore')

# Revisar si hay algún cero
if (table_clean == 0).any().any():
    print("⚠️ Hay al menos una celda con 0 en la tabla de contingencia.")
else:
    print("✅ Todas las celdas son mayores que 0.")

⚠️ Hay al menos una celda con 0 en la tabla de contingencia.


In [68]:
table_clean

Estado de Inmunización,0,1
Diagnóstico VRS,,
0,0,3
1,1,2


In [ ]:

if (0 not in contingency_table.columns) | :
    return 'No hay no inmunizados'

# Extraer valores específicos para el cálculo del odds ratio
a = contingency_table.loc[1, 1]  # Casos expuestos
b = contingency_table.loc[1, 0]  # Casos no expuestos
c = contingency_table.loc[0, 1]  # Controles expuestos
d = contingency_table.loc[0, 0]  # Controles no expuestos


# Calcular el odds ratio y la efectividad
odds_ratio = (a * d) / (b * c)
efectividad = (1 - odds_ratio) * 100

In [ ]:
# match_vars_distance_nn=['edad_relativa','PESO']
# match_vars_exact_nn = ['SEXO','SEMANAS','group','region','muy_prematuro','PREVI','prematuro'] #NOMBRE_REGION
# weights = {'edad_relativa': 1,'PESO': 0.4} ####SIRVE UNICAMENTE PARA NN

# match_vars_distance_IP = ['PESO','edad_relativa','SEMANAS']
# match_vars_distance_IP_francia = ['PESO','edad_relativa','SEMANAS','ingreso_relativo']

# match_vars_exact_IP_previ = ['sexo','region','group','PREVI'] # region
# match_vars_exact_IP_not_previ = ['sexo','region','group']


################# TEST PIPELINE #######################

match_vars_distance_nn=['edad_relativa','ingreso_relativo']
match_vars_exact_nn = ['group','region','PREVI','prematuro'] #NOMBRE_REGION
weights = {'edad_relativa': 1,'ingreso_relativo': 2} 



match_vars_distance_IP = ['edad_relativa']
match_vars_distance_IP_francia = ['edad_relativa','ingreso_relativo']

match_vars_exact_IP_previ = ['prematuro','region','group','PREVI'] # region
match_vars_exact_IP_not_previ = ['prematuro','region','group']


#########################################################


filtro_comba_dict = {  
                    'prematuros': '(prematuro == 1)',
                    'prematuros_seasonal': '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                    'prematuros_catchup': '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                     
                  #  'VRS_general': 'MARCA==0',
                     
                    #  'prematuros_fonasa': '(prematuro == 1)' + ' & ' + '(PREVI == 1.0)',
                    #  'prematuros_isapre': '(prematuro == 1)' + ' & ' + '(PREVI == 2.0)',
                    #  'prematuros_sin_previ': '(prematuro == 1)' + ' & ' + '(PREVI == 96.0)',
                    
                     'displasia': 'RUN.isin(@ruts_displasia.RUN)',
                     'displasia_seasonal': '(RUN.isin(@ruts_displasia.RUN))' + ' & ' + '(group == "SEASONAL")',
                     'displasia_catchup': '(RUN.isin(@ruts_displasia.RUN))' + ' & ' + '(group == "CATCH_UP")',
                     
                     'cardiopats_1': 'RUN.isin(@ruts_cariopaticos_1.RUN)',
                     'cardiopats_1_seasonal': '(RUN.isin(@ruts_cariopaticos_1.RUN))' + ' & ' + '(group == "SEASONAL")',
                     'cardiopats_1_catchup': '(RUN.isin(@ruts_cariopaticos_1.RUN))' + ' & ' + '(group == "CATCH_UP")',
                    #  'cardiopats_2': 'RUN.isin(@ruts_cariopaticos_2.RUN)',

                     'cardiopats_O_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)',
                     'cardiopats_Y_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)',
                     
                    #  'cardiopats_Y_no_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 0)',
                    #  'prematuros_Y_no_cardio': '(prematuro == 1)' + ' & ' + '~RUN.isin(@ruts_cariopaticos_1.RUN)',
                     
                      'cardiopats_O_prematuros_catchup': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                      'cardiopats_O_prematuros_seasonal': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                    #  'cardiopats_Y_prematuros_catchup': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                    #  'cardiopats_Y_prematuros_seasonal': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',

                    #  'rural': 'is_rural == 1'
                    }

df_copy = df.copy()
#df_francia = df.query('(fechaIng_any.notna()) & (diag_1_leter!="J")').copy()
df_francia = df.query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').copy()

# list_experiments_francia_wPrevi = results_match(df_case_study=df_copy,
#                                  df_control_study=df_francia,
#                                  filtros_dic=filtro_comba_dict,
#                                  match_vars_distance_nn=match_vars_distance_nn,
#                                  match_vars_exact_nn=match_vars_exact_nn,
#                                  match_vars_distance_IP=match_vars_distance_IP_francia,
#                                  match_vars_exact_IP=match_vars_exact_IP_previ,
#                                  weights=weights, 
#                                  list_experiments = [],
#                                  nn=False,
#                                  mip=True,
#                                  ratio="1:1",
#                                  covs = ['sexo','SEMANAS','PESO'] )

# df_copy = df.copy()

list_experiments_all_born = results_match(df_case_study=df_copy,
                                 df_control_study=df_copy,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ,
                                 weights=weights, 
                                 list_experiments = [],
                                 nn=False,
                                 ratio="1:4",
                                 covs = ['sexo','PESO','SEMANAS','cardio1','displa']) # ['sexo','SEMANAS','PESO','cardio1','prematuro']


df_copy = df.copy()
#df_francia = df.query('(fechaIng_any.notna()) & (diag_1_leter!="J")').copy()
df_francia = df.query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').copy()

list_experiments_francia_not_previ = results_match(df_case_study=df_copy,
                                 df_control_study=df_francia,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP_francia,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ,
                                 weights=weights, 
                                 list_experiments = [],
                                 nn=False,
                                 ratio="1:1",
                                 covs =  ['sexo','SEMANAS','PESO','cardio1','PREVI','displa'])


prematuros 218 13980 ['sexo', 'PESO', 'SEMANAS', 'cardio1', 'displa']
creacion conjuntos A_i time: 5.85700249671936
creacion variables time: 13.87600302696228
1.7408037222333308


KeyboardInterrupt: 

In [ ]:
# df_final = summary_eff(list_experiments_all_born, list_experiments_francia_wPrevi, list_experiments_francia_not_previ)
df_final = summary_eff_aux(list_experiments_all_born,list_experiments_francia_not_previ)

In [ ]:
with pd.ExcelWriter(path_data/"resultados_comparados_copy.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_final.to_excel(writer, sheet_name="Tabla_2_ratio_1_1_unif")

In [ ]:
copia_born = list_experiments_all_born.copy()
copia_france_previ= list_experiments_francia_wPrevi.copy()
copia_france_not_previ = list_experiments_francia_not_previ.copy()

lista_problemas_AB = [dic for dic in copia_born if len(dic) == 3]
lista_problemas_francia_previ = [dic for dic in copia_france_previ if len(dic) == 3]
lista_problemas_francia_not_previ = [dic for dic in copia_france_not_previ if len(dic) == 3]

for d in lista_problemas_AB:
    d['controles'] = 'all_born'

for d in lista_problemas_francia_previ:
    d['controles'] = 'francia_previ'

for d in lista_problemas_francia_not_previ:
    d['controles'] = 'francia_not_previ'
    
df_problemas = pd.DataFrame(lista_problemas_AB + lista_problemas_francia_previ + lista_problemas_francia_not_previ)
df_problemas

In [ ]:
with pd.ExcelWriter(path_data/"resultados_comparados_copy.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_problemas.to_excel(writer, sheet_name="df_problemas",index=False)

In [179]:
df.query('(prematuro == 1)' + ' & ' + '(group == "SEASONAL")').groupby('inmunizado').event.value_counts().unstack(-1)

event,0,1
inmunizado,,
0,302,1
1,6839,90


In [ ]:
# match_vars_distance_nn=['edad_relativa','PESO']
# match_vars_exact_nn = ['SEXO','SEMANAS','group','region','muy_prematuro','PREVI','prematuro'] #NOMBRE_REGION
# weights = {'edad_relativa': 1,'PESO': 0.4} ####SIRVE UNICAMENTE PARA NN

match_vars_distance_IP = ['PESO','edad_relativa','SEMANAS']

match_vars_exact_IP_previ = ['sexo','region','group','PREVI'] # region


dict_test = {
                    #  'prematuros': '(prematuro == 1)',
                    #  'prematuros_seasonal': '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                    #  'prematuros_catchup': '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                     
                    #  # 'VRS_general': ''
                     
                    #  'prematuros_fonasa': '(prematuro == 1)' + ' & ' + '(PREVI == 1.0)',
                    #  'prematuros_isapre': '(prematuro == 1)' + ' & ' + '(PREVI == 2.0)',
                    #  'prematuros_sin_previ': '(prematuro == 1)' + ' & ' + '(PREVI == 96.0)',
                     
                     'cardiopats_1': 'RUN.isin(@ruts_cariopaticos_1.RUN)',
                    #  'cardiopats_2': 'RUN.isin(@ruts_cariopaticos_2.RUN)',

                    #  'cardiopats_O_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)',
                    #  'cardiopats_Y_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)',
                     
                    #  'cardiopats_Y_no_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 0)',
                    #  'prematuros_Y_no_cardio': '(prematuro == 1)' + ' & ' + '~RUN.isin(@ruts_cariopaticos_1.RUN)',
                     
                    #  'cardiopats_O_prematuros_catchup': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                    #  'cardiopats_O_prematuros_seasonal': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                    #  'cardiopats_Y_prematuros_catchup': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                    #  'cardiopats_Y_prematuros_seasonal': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                     
                    #  'rural': 'is_rural == 1'
                    }


df_copy = df.copy()

lists_to35 = results_match(df_case_study=df_copy,
                            df_control_study=df_copy,
                            filtros_dic=dict_test,
                            match_vars_distance_nn=match_vars_distance_nn,
                            match_vars_exact_nn=match_vars_exact_nn,
                            match_vars_distance_IP=match_vars_distance_IP,
                            match_vars_exact_IP=match_vars_exact_IP_not_previ,
                            weights=weights, 
                            list_experiments = [],
                            nn=False,
                            mip=True)

cardiopats_1 41 1150
creacion conjuntos A_i time: 0.11756396293640137
creacion variables time: 2.511113166809082
5.1761686613594176
optimize model 1 time: 0.006411075592041016
optimize model 2 time: 0.011758804321289062
matched_data time: 0.09012532234191895
Total cases = 41, Total controls = 1150
Total cases matched is : 41, Total control matched is : 205
ratio: 1:5

Odds Ratios y Efectividad:
                      Coeficientes        OR  Efectividad      0.975      0.025
estado_inmunizacion      -2.03119  0.131179    86.882064  96.800953  46.208902

IC disntace: 2.822255027454238


In [ ]:
################################################################### UPC ####################################################################################

df_upc = (df_upc_match_case
      .drop(columns=[col for col in df_upc_match_case.columns if ((col.startswith('inm_')) | (col.startswith('MES_')) | (col.startswith('DIA_')) | 
                                               (col.startswith('SERC_')) | ((col.startswith('ANO_') & (col.endswith('TRAS')))) | 
                                               (col.startswith('AREAF')) | (col.startswith('dias_en')) | (col.startswith('fecha_tras')) |
                                               (col.endswith('Dall')) )])
      .copy()
)

match_vars_distance_nn=['edad_relativa','ingreso_relativo']
match_vars_exact_nn = ['group','region','PREVI','prematuro'] #NOMBRE_REGION
weights = {'edad_relativa': 1,'ingreso_relativo': 2} 



match_vars_distance_IP = ['edad_relativa']
match_vars_distance_IP_francia = ['edad_relativa','ingreso_relativo']

match_vars_exact_IP_previ = ['prematuro','region','group','PREVI'] # region
match_vars_exact_IP_not_previ = ['prematuro','region','group']

filtro_comba_dict = {  
                  #  'prematuros': '(prematuro == 1)',
                  #  'prematuros_seasonal': '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                  #  'prematuros_catchup': '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                     
                    'UPC_general': 'MARCA==0',
                     
                    #  'prematuros_fonasa': '(prematuro == 1)' + ' & ' + '(PREVI == 1.0)',
                    #  'prematuros_isapre': '(prematuro == 1)' + ' & ' + '(PREVI == 2.0)',
                    #  'prematuros_sin_previ': '(prematuro == 1)' + ' & ' + '(PREVI == 96.0)',
                     
                    # 'cardiopats_1': 'RUN.isin(@ruts_cariopaticos_1.RUN)',
                    # 'cardiopats_1_seasonal': '(RUN.isin(@ruts_cariopaticos_1.RUN))' + ' & ' + '(group == "SEASONAL")',
                    # 'cardiopats_1_catchup': '(RUN.isin(@ruts_cariopaticos_1.RUN))' + ' & ' + '(group == "CATCH_UP")',
                    #  'cardiopats_2': 'RUN.isin(@ruts_cariopaticos_2.RUN)',

                    # 'cardiopats_O_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)',
                    # 'cardiopats_Y_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)',
                     
                    #  'cardiopats_Y_no_prema': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 0)',
                    #  'prematuros_Y_no_cardio': '(prematuro == 1)' + ' & ' + '~RUN.isin(@ruts_cariopaticos_1.RUN)',
                     
                    #  'cardiopats_O_prematuros_catchup': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                    #  'cardiopats_O_prematuros_seasonal': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' | ' + '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',
                    #  'cardiopats_Y_prematuros_catchup': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)' + ' & ' + '(group == "CATCH_UP")',
                    #  'cardiopats_Y_prematuros_seasonal': 'RUN.isin(@ruts_cariopaticos_1.RUN)' + ' & ' + '(prematuro == 1)' + ' & ' + '(group == "SEASONAL")',

                    #  'rural': 'is_rural == 1'
                    }


df_copy = df_upc.copy()

list_experiments_all_born = results_match(df_case_study=df_copy,
                                 df_control_study=df_copy,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ,
                                 weights=weights, 
                                 list_experiments = list_experiments_all_born,
                                 nn=False,
                                 ratio="1:1",
                                 covs = ['sexo','PESO','SEMANAS','cardio1'])


df_copy = df_upc.copy()
# df_francia = df.query('(fechaIng_any.notna()) & (diag_1_leter!="J")').copy()
df_francia = df_upc.query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').copy()

list_experiments_francia_not_previ = results_match(df_case_study=df_copy,
                                 df_control_study=df_francia,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP_francia,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ,
                                 weights=weights, 
                                 list_experiments = list_experiments_francia_not_previ,
                                 nn=False,
                                 ratio="1:1",
                                 covs =  ['sexo','SEMANAS','PESO','cardio1','PREVI'])

# list_experiments_francia_wPrevi = results_match(df_case_study=df_copy,
#                                  df_control_study=df_francia,
#                                  filtros_dic=filtro_comba_dict,
#                                  match_vars_distance_nn=match_vars_distance_nn,
#                                  match_vars_exact_nn=match_vars_exact_nn,
#                                  match_vars_distance_IP=match_vars_distance_IP_francia,
#                                  match_vars_exact_IP=match_vars_exact_IP_previ,
#                                  weights=weights, 
#                                  list_experiments = list_experiments_francia_wPrevi,
#                                  nn=False,
#                                  mip=True)

# df_copy = df.copy()

In [12]:
marcel_summary_all_born, all_results_df_all_born = tabla_marcel(list_experiments_all_born)
enes_matches_all_born, all_n_eff_all_born  = tabla_final(all_results_df_all_born,marcel_summary_all_born)
df_all_born = all_n_eff_all_born#.query('efectividad != "na"')
#all_n_eff_all_born#.query('efectividad != "na"')
marcel_summary_francia_wPrevi, all_results_df_francia_wPrevi = tabla_marcel(list_experiments_francia_wPrevi)
enes_matches_francia_wPrevi, all_n_eff_francia_wPrevi  = tabla_final(all_results_df_francia_wPrevi,marcel_summary_francia_wPrevi)
df_francia_previ = all_n_eff_francia_wPrevi#.query('efectividad != "na"')
#all_n_eff_francia_wPrevi#.query('efectividad != "na"')
marcel_summary_francia_not_previ, all_results_df_francia_not_previ = tabla_marcel(list_experiments_francia_not_previ)
enes_matches_francia_not_previ, all_n_eff_francia_not_previ  = tabla_final(all_results_df_francia_not_previ,marcel_summary_francia_not_previ)
df_francia_not_previ = all_n_eff_francia_not_previ#.query('efectividad != "na"')
#all_n_eff_francia_not_previ#.query('efectividad != "na"')

# Paso 1: Asegúrate de que el índice sea 'filtro' para todas
df_all_born_macro = df_all_born.set_index("filtro")
df_francia_previ_macro = df_francia_previ.set_index("filtro")
df_francia_not_previ_macro = df_francia_not_previ.set_index("filtro")

# Paso 2: Renombrar columnas con MultiIndex
df_all_born_macro.columns = pd.MultiIndex.from_product([["all_born"], df_all_born_macro.columns])
df_francia_previ_macro.columns = pd.MultiIndex.from_product([["francia_previ"], df_francia_previ_macro.columns])
df_francia_not_previ_macro.columns = pd.MultiIndex.from_product([["francia_not_previ"], df_francia_not_previ_macro.columns])

# Paso 3: Concatenar horizontalmente por el índice (filtro)
df_final = pd.concat([df_all_born_macro, df_francia_not_previ_macro, df_francia_previ_macro], axis=1)
df_final.columns = pd.MultiIndex.from_tuples([
    (nivel_superior, "prop" if nivel_inferior == "prop_cases_match" else nivel_inferior)
    for nivel_superior, nivel_inferior in df_final.columns
])
df_final.columns = pd.MultiIndex.from_tuples([
    (nivel_superior, "matched" if nivel_inferior == "cases_matched" else nivel_inferior)
    for nivel_superior, nivel_inferior in df_final.columns
])
df_final.columns = pd.MultiIndex.from_tuples([
    (nivel_superior, "cases" if nivel_inferior == "total_cases" else nivel_inferior)
    for nivel_superior, nivel_inferior in df_final.columns
])
df_final

all_born                        francia_not_previ  \
                      efectividad cases matched prop           efectividad   
filtro                                                                       
UPC_general  90.67 (84.54; 94.37)   268     268  1.0  91.37 (85.55; 94.85)   

                                        francia_previ                      
            cases matched  prop           efectividad cases matched  prop  
filtro                                                                     
UPC_general   268     266  0.99  91.43 (85.43; 94.96)   268     238  0.89

In [13]:
with pd.ExcelWriter(path_data/"resultados_comparados_copy.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_final.to_excel(writer, sheet_name="upc_copypaste")

In [ ]:
match_vars_distance_nn=['edad_relativa','PESO']
match_vars_exact_nn = ['SEXO','SEMANAS','group','region','muy_prematuro','PREVI','prematuro'] #NOMBRE_REGION
weights = {'edad_relativa': 1,'PESO': 0.4} ####SIRVE UNICAMENTE PARA NN

match_vars_distance_IP = ['PESO','edad_relativa','SEMANAS']
match_vars_distance_IP_francia = ['PESO','edad_relativa','SEMANAS','ingreso_relativo']

match_vars_exact_IP_previ = ['sexo','region','group','PREVI'] # region
match_vars_exact_IP_not_previ = ['sexo','region','group']

filtro_comba_dict = {'VRS_general': 'MARCA==0'}

df_copy = df.copy()
#df_francia = df.query('(fechaIng_any.notna()) & (diag_1_leter!="J")').copy()
df_francia = df.query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').copy()

list_experiments_francia_wPrevi = results_match(df_case_study=df_copy,
                                 df_control_study=df_francia,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP_francia,
                                 match_vars_exact_IP=match_vars_exact_IP_previ,
                                 weights=weights, 
                                 #list_experiments = list_experiments_francia_wPrevi,
                                 nn=False,
                                 mip=True)

# df_copy = df.copy()

list_experiments_all_born = results_match(df_case_study=df_copy,
                                 df_control_study=df_copy,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ,
                                 weights=weights, 
                                # list_experiments = list_experiments_all_born,
                                 nn=False,
                                 mip=True)


# df_copy = df.copy()
# df_francia = df.query('(fechaIng_any.notna()) & (diag_1_leter!="J")').copy()

list_experiments_francia_not_previ = results_match(df_case_study=df_copy,
                                 df_control_study=df_francia,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP_francia,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ,
                                 weights=weights, 
                                # list_experiments = list_experiments_francia_not_previ,
                                 nn=False,
                                 mip=True)

VRS_general 1275 1451
creacion conjuntos A_i time: 2.8790013790130615
Set parameter Username
Academic license - for non-commercial use only - expires 2025-12-09
creacion variables time: 41.357508420944214
12.288243292101813
optimize model 1 time: 460.21267676353455


# Intento T1

In [ ]:
df_case = df.query('fechaIng_vrs.notna()').query("diag_vrs==True").copy()
df_control = df.query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').query("diag_vrs==False").copy()

distance_vars = ['edad_relativa','SEMANAS','ingreso_relativo'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls, cases, controls, ratio, n_control_matched, n_case_matched = charly_double_mip(df_cases=df_case,df_control=df_control, distance_vars=distance_vars, exact_var_match = ['region','group'], ratio="1:3")


creacion conjuntos A_i time: 2.2515809535980225
Set parameter Username
Academic license - for non-commercial use only - expires 2025-12-09
creacion variables time: 142.78150701522827
12.259510369177278


In [3]:
df_case = df.query('fechaIng_vrs.notna()').query("diag_vrs==True").copy()
df_control = df.query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').query("diag_vrs==False").copy()

weights = {'edad_relativa': 1,'ingreso_relativo':2 }

matched_data, matched_incompleto, matched_controls, cases, controls, ratio, n_control_matched, n_case_matched = match_nn_max_dist_weigths(df_control, df_case,
                                                                                                                                    match_vars_nn=['edad_relativa','ingreso_relativo'], 
                                                                                                                                    match_vars_exact = ['SEMANAS','group','region'],
                                                                                                                                    match_vars_onehot=[],
                                                                                                                                    ratio="1:1",
                                                                                                                                    max_distance=5,
                                                                                                                                    weights=weights)

display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['ingreso_relativo','SEMANAS','edad_relativa']))


df_matched = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched,cases, controls, ratio, n_control_matched, n_case_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Total cases = 1275, Total controls = 1451
Total cases matched is : 672, Total control matched is : 672
ratio: 1:1
No matched : 603


,Columna,Media_df1,Media_df2,T-stat,P-value,Mensaje
0,ingreso_relativo,89.38,98.80,-3.15,0.00,Existe una diferencia significativa.
1,SEMANAS,38.04,38.04,0.00,1.00,
2,edad_relativa,158.08,147.08,2.27,0.02,Existe una diferencia significativa.




Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.843882
1    0.426378
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización    0     1   All
Diagnóstico VRS                        
0                        37   635   672
1                       200   472   672
All                     237  1107  1344

Tabla de Porcentajes:
 Estado de Inmunización           0          1    All
Diagnóstico VRS                                     
0                        15.611814   57.36224   50.0
1                        84.388186   42.63776   50.0
All                     100.000000  100.00000  100.0

Odds Ratio:  0.14
Efectividad:  86.2 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               diag_vrs   No. Observations:                 1344
Model:               ConditionalLogit   No. groups:                        672
Log-Likelihood:               -38

In [20]:
df_matched = df_matched.assign(cardio1 = lambda x: x.RUN.isin(ruts_cariopaticos_1.RUN.unique()).astype(int))

In [39]:
leo_zonas_rename = {'Macrozona Centro Sur':'South Macrozone',
                    'Macrozona Sur':'South Macrozone',
                    'Macrozona Norte':'North Macrozone',
                    'Macrozona Centro Norte':'Central Macrozone',
                    'Macrozona Austral':'Austral Macrozone'}

region_to_macrozone_agencia = {
    "ARICA Y PARINACOTA": "Macrozona Norte",
    "TARAPACA": "Macrozona Norte",
    "ANTOFAGASTA": "Macrozona Norte",
    "ATACAMA": "Macrozona Norte",
    "COQUIMBO": "Macrozona Centro Norte",
    "VALPARAISO": "Macrozona Centro Norte",
    "METROPOLITANA": "Macrozona Centro Norte",
    "O'HIGGINS": "Macrozona Centro Norte",
    "MAULE": "Macrozona Centro Sur",
    "NUBLE": "Macrozona Centro Sur",
    "BIOBIO": "Macrozona Centro Sur",
    "ARAUCANIA": "Macrozona Centro Sur",
    "LOS RIOS": "Macrozona Sur",
    "LOS LAGOS": "Macrozona Sur",
    "AISEN": "Macrozona Austral",
    "MAGALLANES Y ANTARTICA": "Macrozona Austral"
}

df_matched["Macrozones"] = df_matched["region"].map(region_to_macrozone_agencia).replace(leo_zonas_rename)

In [63]:
def tabla_1_match(df, id_col='RUN', group_col='Group', case_label='Caso', control_label='Control', decimales=2):
    import pandas as pd
    import numpy as np

    df = df.copy()
    df['grupo_tabla'] = np.where(df[id_col] == df[group_col], case_label, control_label)
    df['sexo'] = df['sexo'].map({0: 'Female', 1: 'Male'})

    variables_numericas_mediana = ['edad_relativa']
    variables_numericas_media = ['SEMANAS', 'PESO']
    variables_categoricas = ['sexo']
    macrozona_var = 'Macrozones'

    etiquetas_variables = {
        'edad_relativa': 'Age (weeks)',
        'SEMANAS': 'Gestational age at birth (weeks)',
        'PESO': 'Birth weight (grams)',
        'sexo': 'Sex',
        'Macrozones': 'Macro-zone',
        'cario1': 'Congenital heart disease',
        'prematuro': 'Prematurity',
        'event_upc': 'PICU admission'
    }

    grupos = df['grupo_tabla'].unique()
    tabla = []
    index = []
    n_por_grupo = {g: len(df[df['grupo_tabla'] == g]) for g in grupos}
    nombres_columnas = {g: f"{g} (n = {n_por_grupo[g]})" for g in grupos}

    for var in variables_numericas_mediana:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            mediana = subset.median()
            iqr = subset.quantile(0.75) - subset.quantile(0.25)
            fila[nombres_columnas[g]] = f'{mediana:.{decimales}f} ({iqr:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

    for var in variables_categoricas:
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append({})  # Fila vacía
        niveles = df[var].dropna().unique()
        for val in niveles:
            fila = {}
            for g in grupos:
                subset = df[df['grupo_tabla'] == g]
                total = len(subset)
                cuenta = (subset[var] == val).sum()
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
            index.append((etiquetas_variables.get(var, var), val))
            tabla.append(fila)

    for var in variables_numericas_media:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g][var]
            media = subset.mean()
            sd = subset.std()
            fila[nombres_columnas[g]] = f'{media:.{decimales}f} ({sd:.{decimales}f})'
        index.append((etiquetas_variables.get(var, var), ''))
        tabla.append(fila)

    index.append(('Risk groups', ''))
    tabla.append({})
    for var in ['cario1', 'prematuro']:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[var] == 1).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append(('Risk groups', etiquetas_variables.get(var, var)))
        tabla.append(fila)

    index.append((etiquetas_variables.get(macrozona_var, macrozona_var), ''))
    tabla.append({})
    niveles = df[macrozona_var].dropna().unique()
    for val in niveles:
        fila = {}
        for g in grupos:
            subset = df[df['grupo_tabla'] == g]
            total = len(subset)
            cuenta = (subset[macrozona_var] == val).sum()
            porcentaje = 100 * cuenta / total if total > 0 else 0
            fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        index.append((etiquetas_variables.get(macrozona_var, macrozona_var), val))
        tabla.append(fila)

    fila = {}
    for g in grupos:
        subset = df[df['grupo_tabla'] == g]
        total = len(subset)
        cuenta = (subset['event_upc'] == 1).sum()
        porcentaje = 100 * cuenta / total if total > 0 else 0
        fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
    index.append((etiquetas_variables.get('event_upc', 'PICU'), ''))
    tabla.append(fila)

    diag_labels = {
        'A099': 'Other gastroenteritis and colitis',
        'A090': 'Infectious diarrhoea',
        'R681': 'Other general symptoms and signs',
        'R11X': 'Nausea and vomiting',
        'R104': 'Abdominal pain',
        'S099': 'Other head injuries',
        'R634': 'Feeding difficulties',
        'R633': 'Polydipsia',
        'N390': 'Urinary tract infection',
        'P599': 'Other neonatal jaundice'
    }

    diag_frecuencias = []
    for diag_code, label in diag_labels.items():
        fila = {}
        count_control = 0
        for g in grupos:
            if g == case_label:
                fila[nombres_columnas[g]] = '-'  # No diagnóstico para los casos
            else:
                subset = df[(df['grupo_tabla'] == g) & (df['DIAG1'] == diag_code)]
                total = len(df[df['grupo_tabla'] == g])
                cuenta = len(subset)
                count_control = cuenta
                porcentaje = 100 * cuenta / total if total > 0 else 0
                fila[nombres_columnas[g]] = f'{cuenta} ({porcentaje:.1f}%)'
        diag_frecuencias.append((count_control, ('Reason for hospital visit (controls)', label), fila))

    index.append(('Reason for hospital visit (controls)', ''))
    tabla.append({})
    for _, idx, fila in sorted(diag_frecuencias, key=lambda x: x[0], reverse=True):
        index.append(idx)
        tabla.append(fila)

    multiindex = pd.MultiIndex.from_tuples(index, names=['Variable', 'Value'])
    return pd.DataFrame(tabla, index=multiindex)



tabla1 = tabla_1_match(df_matched)
display(tabla1)

Caso (n = 672)  \
Variable                             Value                                                 
Age (weeks)                                                              157.50 (128.25)   
Sex                                                                                  NaN   
                                     Male                                    419 (62.4%)   
                                     Female                                  253 (37.6%)   
Gestational age at birth (weeks)                                            38.04 (1.46)   
Birth weight (grams)                                                    3275.76 (498.70)   
Risk groups                                                                          NaN   
                                     Congenital heart disease                  13 (1.9%)   
                                     Prematurity                              86 (12.8%)   
Macro-zone                                                                           NaN   
                                     Central Macrozone                       407 (60.6%)   
                                     South Macrozone                         214 (31.8%)   
                                     North Macrozone                           46 (6.8%)   
                                     Austral Macrozone                          5 (0.7%)   
PICU admission                                                               138 (20.5%)   
Reason for hospital visit (controls)                                                 NaN   
                                     Urinary tract infection                           -   
                                     Other neonatal jaundice                           -   
                                     Other gastroenteritis and colitis                 -   
                                     Infectious diarrhoea                              -   
                                     Other general symptoms and signs                  -   
                                     Nausea and vomiting                               -   
                                     Abdominal pain                                    -   
                                     Other head injuries                               -   
                                     Feeding difficulties                              -   
                                     Polydipsia                                        -   

                                                                       Control (n = 672)  
Variable                             Value                                                
Age (weeks)                                                              158.00 (154.25)  
Sex                                                                                  NaN  
                                     Male                                    407 (60.6%)  
                                     Female                                  265 (39.4%)  
Gestational age at birth (weeks)                                            38.04 (1.46)  
Birth weight (grams)                                                    3298.55 (485.61)  
Risk groups                                                                          NaN  
                                     Congenital heart disease                   3 (0.4%)  
                                     Prematurity                              86 (12.8%)  
Macro-zone                                                                           NaN  
                                     Central Macrozone                       407 (60.6%)  
                                     South Macrozone                         214 (31.8%)  
                                     North Macrozone                           46 (6.8%)  
                                     Austral Macrozone                          5 (0.7%)  
PICU admission                           

In [64]:
with pd.ExcelWriter(path_data/"resultados_comparados_copy.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    tabla1.to_excel(writer, sheet_name="tabla_1")

In [68]:
matched_data.assign(who_is = lambda x: np.where(x.diag_1_leter!='J','control','case')).groupby('who_is').cama.value_counts()

who_is   cama
case     UPC     138
control  UPC     122
Name: count, dtype: int64

In [70]:
results = analyze_vrs_data(df_matched.assign(who_is = lambda x: np.where(x.diag_1_leter!='J','control','case'),
                                             cama_bin = lambda x: (x.cama.notna() & (x.who_is=='case')).astype(int)),cases, controls, ratio, n_control_matched, n_case_matched)

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

Tabla de Contingencia:
 Estado de Inmunización    0     1   All
Diagnóstico VRS                        
0                       190  1016  1206
1                        47    91   138
All                     237  1107  1344

Tabla de Porcentajes:
 Estado de Inmunización           0           1         All
Diagnóstico VRS                                           
0                        80.168776   91.779584   89.732143
1                        19.831224    8.220416   10.267857
All                     100.000000  100.000000  100.000000

Odds Ratio:  0.36
Efectividad:  63.8 %

Resumen del modelo:
                   Conditional Logit Model Regression Results                  
Dep. Variable:               cama_bin   No. Observations:                  276
Model:               ConditionalLogit   No. groups:                        138
Log-Likelihood:               -77.360   Min group size:                      2
Method:                          BFGS   Max group size:                      2


In [ ]:


match_vars_distance_nn=['edad_relativa','ingreso_relativo']
match_vars_exact_nn = ['group','region','PREVI','prematuro'] #NOMBRE_REGION
weights = {'edad_relativa': 1,'ingreso_relativo': 2} 



match_vars_distance_IP = ['edad_relativa']
match_vars_distance_IP_francia = ['edad_relativa','ingreso_relativo']

match_vars_exact_IP_previ = ['prematuro','region','group','PREVI'] # region
match_vars_exact_IP_not_previ = ['prematuro','region','group']

filtro_comba_dict = { 'prematuros': '(prematuro == 1)'}

df_copy = df.copy()
df_francia = df.query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').copy()

list_experiments_francia_wPrevi = results_match(df_case_study=df_copy,
                                 df_control_study=df_francia,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP_francia,
                                 match_vars_exact_IP=match_vars_exact_IP_previ,
                                 weights=weights, 
                                 list_experiments = [],
                                 nn=False,
                                 mip=True,
                                 ratio="1:1",
                                 covs = ['sexo','SEMANAS','PESO'] )

# df_copy = df.copy()

list_experiments_all_born = results_match(df_case_study=df_copy,
                                 df_control_study=df_copy,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ,
                                 weights=weights, 
                                 list_experiments = [],
                                 nn=False,
                                 ratio="1:5",
                                 covs = ['sexo','SEMANAS','PESO'])


# df_copy = df.copy()
# df_francia = df.query('(fechaIng_any.notna()) & (diag_1_leter!="J")').copy()

list_experiments_francia_not_previ = results_match(df_case_study=df_copy,
                                 df_control_study=df_francia,
                                 filtros_dic=filtro_comba_dict,
                                 match_vars_distance_nn=match_vars_distance_nn,
                                 match_vars_exact_nn=match_vars_exact_nn,
                                 match_vars_distance_IP=match_vars_distance_IP_francia,
                                 match_vars_exact_IP=match_vars_exact_IP_not_previ,
                                 weights=weights, 
                                 list_experiments = [],
                                 nn=False,
                                 ratio="1:1",
                                 covs = ['sexo','SEMANAS','PESO'])

In [3]:
df_case.cardio1.value_counts()

AttributeError: 'DataFrame' object has no attribute 'cardio1'

In [13]:
toma = [col for col in df.columns if col.startswith('fecha')]

In [14]:
toma

['fecha_nac',
 'fechaIng_any',
 'fechaEgr',
 'fechaInm',
 'fechaIng_vrs',
 'fechaIng_LRTI',
 'fecha_upc',
 'fecha_upc_vrs',
 'fechaIng_vrs_copy']

In [31]:
df[['edad_relativa','ingreso_relativo']].isna().sum()

edad_relativa            0
ingreso_relativo    142986
dtype: int64

In [32]:
df_case = df.assign(log_peso = lambda x: np.log(x.PESO)).query('fechaIng_vrs.notna()').query("diag_vrs==True").query('(prematuro == 1)').copy()
df_control = df.assign(log_peso = lambda x: np.log(x.PESO)).query('(prematuro == 1)').query("diag_vrs==False").copy() # .query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))')

print(df_case.RUN.nunique(), df_control.RUN.nunique())
distance_vars = ['edad_relativa'] # ,'ingreso_relativo'

matched_data, feasible_controls, cases, controls, ratio, n_control_matched, n_case_matched = charly_double_mip(df_cases=df_case,
                                                                                                               df_control=df_control, 
                                                                                                               distance_vars=distance_vars, 
                                                                                                               exact_var_match = ['region','group','prematuro'], ratio="1:5")
#display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa','ingreso_relativo']))

df_matched_prterms = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched_prterms.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched_prterms, cases, controls,n_control_matched, n_case_matched, ratio,covs=['sexo','SEMANAS','PESO','cardio1'])

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

226 14689
creacion conjuntos A_i time: 8.383785009384155
creacion variables time: 21.329331159591675
1.7478653518383114
optimize model 1 time: 4.288828611373901
optimize model 2 time: 4.4849982261657715
matched_data time: 2.397756338119507
Total cases = 226, Total controls = 14689
Total cases matched is : 226, Total control matched is : 1130
ratio: 1:5


Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.430380
1    0.150352
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización   0     1   All
Diagnóstico VRS                       
0                       45  1085  1130
1                       34   192   226
All                     79  1277  1356

Tabla de Porcentajes:
 Estado de Inmunización           0           1         All
Diagnóstico VRS                                           
0                        56.962025   84.964761   83.333333
1                        43.037975   15.035239   16.666667
All        

In [36]:
df_case = df.assign(log_peso = lambda x: np.log(x.PESO)).query('fechaIng_vrs.notna()').query("diag_vrs==True").copy()
df_control = df.assign(log_peso = lambda x: np.log(x.PESO)).query("diag_vrs==False").copy() # .query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))')

print(df_case.RUN.nunique(), df_control.RUN.nunique())

1275 152491


In [35]:
df_case = df.assign(log_peso = lambda x: np.log(x.PESO)).query('fechaIng_vrs.notna()').query("diag_vrs==True").copy()
df_control = df.assign(log_peso = lambda x: np.log(x.PESO)).query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))').query("diag_vrs==False").copy() # .query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))')

print(df_case.RUN.nunique(), df_control.RUN.nunique())
distance_vars = ['edad_relativa','ingreso_relativo'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls, cases, controls, ratio, n_control_matched, n_case_matched = charly_double_mip(df_cases=df_case,
                                                                                                               df_control=df_control, 
                                                                                                               distance_vars=distance_vars, 
                                                                                                               exact_var_match = ['region','group','prematuro'], ratio="1:1")
#display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa','ingreso_relativo']))

df_matched_prterms = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched_prterms.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched_prterms, cases, controls,n_control_matched, n_case_matched, ratio,covs=['sexo','SEMANAS','PESO','cardio1'])

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])

1275 1741
creacion conjuntos A_i time: 5.694754362106323
creacion variables time: 222.53283548355103
4.392122577481735
optimize model 1 time: 1.318249225616455
optimize model 2 time: 4.101908922195435
matched_data time: 2.2190494537353516
Total cases = 1275, Total controls = 1741
Total cases matched is : 1171, Total control matched is : 1171
ratio: 1:1


Dado tu estado de inmunizacion cual es probabilidad de estar contagiado
estado_inmunizacion
0    0.850267
1    0.433435
Name: diag_vrs, dtype: float64


Tabla de Contingencia:
 Estado de Inmunización    0     1   All
Diagnóstico VRS                        
0                        56  1115  1171
1                       318   853  1171
All                     374  1968  2342

Tabla de Porcentajes:
 Estado de Inmunización           0           1    All
Diagnóstico VRS                                      
0                        14.973262   56.656504   50.0
1                        85.026738   43.343496   50.0
All                     10

In [ ]:
df_case = df.assign(log_peso = lambda x: np.log(x.PESO)).query('fechaIng_vrs.notna()').query("diag_vrs==True").copy()
df_control = df.assign(log_peso = lambda x: np.log(x.PESO)).query("diag_vrs==False").copy() # .query('(fechaIng_any.notna()) & (DIAG1.isin(@icd10_codes_fr))')

print(df_case.RUN.nunique(), df_control.RUN.nunique())
distance_vars = ['edad_relativa'] #[,'COMUNA_N' 'ESTAB'

matched_data, feasible_controls, cases, controls, ratio, n_control_matched, n_case_matched = charly_double_mip(df_cases=df_case,
                                                                                                               df_control=df_control, 
                                                                                                               distance_vars=distance_vars, 
                                                                                                               exact_var_match = ['region','group','prematuro'], ratio="1:10")
#display(comparar_medias_test(matched_data[~matched_data.Matched_Case_RUN.isna()],matched_data[matched_data.Matched_Case_RUN.isna()],['PESO','SEMANAS','SEXO','edad_relativa','ingreso_relativo']))

df_matched_prterms = matched_data.copy()
print('\n')
print("Dado tu estado de inmunizacion cual es probabilidad de estar contagiado")
print(df_matched_prterms.groupby('estado_inmunizacion')['diag_vrs'].mean())

print('\n')
results = analyze_vrs_data(df_matched_prterms, cases, controls,n_control_matched, n_case_matched, ratio,covs=['sexo','SEMANAS','PESO','cardio1'])

print("Tabla de Contingencia:\n", results['contingency_table'])
print("\nTabla de Porcentajes:\n", results['percentage_table'])
print(f"\nOdds Ratio: {results['odds_ratio']: .2}")
print(f"Efectividad: {results['efectividad']: .3} %")
print("\nResumen del modelo:\n", results['model_summary'])
print("\nOdds Ratios y Efectividad:\n", results['odds_ratios'])